In [1]:
import torch

input = torch.tensor([1.0, 1.0], requires_grad=False)
w1 = torch.tensor([2.0, 2.0], requires_grad=True)
w2 = torch.tensor([3.0, 3.0], requires_grad=True)
w3 = torch.tensor([4.0, 4.0], requires_grad=True)

l1 = input * w1
l2 = l1 + w2
l3 = l1 * w3
l4 = l2 * l3
loss = torch.mean(l4)

l1.retain_grad()
l2.retain_grad()
l3.retain_grad()
l4.retain_grad()
loss.retain_grad()

loss.backward()

print(loss.grad)  # tensor(1.)
print(l4.grad)  # tensor([0.5000, 0.5000])
print(l3.grad)  # tensor([2.5000, 2.5000])
print(w3.grad)  # tensor([5., 5.])
print(l2.grad)  # tensor([4., 4.])
print(w2.grad)  # tensor([4., 4.])

tensor(1.)
tensor([0.5000, 0.5000])
tensor([2.5000, 2.5000])
tensor([5., 5.])
tensor([4., 4.])
tensor([4., 4.])


In [ ]:
# 计算图中的结点实际上是 grad_fn, 不是由参数构成的
l4.grad_fn.next_functions  # 指针指向前两个 grad_fn

((<AddBackward0 at 0x253c020e170>, 0), (<MulBackward0 at 0x253c020dd80>, 0))

In [3]:
# 尝试一下钩子函数
input = torch.tensor([1.0, 1.0], requires_grad=False)
w1 = torch.tensor([2.0, 2.0], requires_grad=True)
w2 = torch.tensor([3.0, 3.0], requires_grad=True)
w3 = torch.tensor([4.0, 4.0], requires_grad=True)

l1 = input * w1
l2 = l1 + w2
l3 = l1 * w3
l4 = l2 * l3

l2.register_hook(lambda grad: print(grad))  # 在收到上游回传的梯度时先打印再更新 l2.grad

loss = torch.mean(l4)
print(loss)
loss.retain_grad()
loss.backward()

tensor(40., grad_fn=<MeanBackward0>)
tensor([4., 4.])


In [4]:
a = torch.tensor([1.0, 3.0], requires_grad=True)
b = a + 2
print(b._version)  # 0

loss = (b * b).mean()
b[0] = 1000.0  # 原地修改,版本+1
print(b._version)  # 1

loss.backward()

0
1


RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [2]], which is output 0 of struct torch::autograd::CopySlices, is at version 1; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [5]:
a = torch.tensor([10.0, 5.0, 2.0, 3.0], requires_grad=True)
print(a, a.is_leaf)
# tensor([10.,  5.,  2.,  3.], requires_grad=True) True

# 叶子节点在使用之前完全不允许进行 inplace 修改(torch 不知道你到底是否需要把这个操作算作计算图中)

a[:] = 0
print(a, a.is_leaf)
# tensor([0., 0., 0., 0.], grad_fn=<CopySlices>) False

loss = (a * a).mean()
loss.backward()
# RuntimeError: leaf variable has been moved into the graph interior

tensor([10.,  5.,  2.,  3.], requires_grad=True) True


RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.